<a href="https://colab.research.google.com/github/Thomas-Fabbris/MMIP-polimi/blob/main/Assignments/lecture_9_OMP_Denoising.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook setup

Import the necessary modules, including `google.colab.drive` for accessing the required assets for the notebook (`Lena512.png`) from Google Drive

In [ ]:
from skimage.io import imread          # pyright: ignore[reportMissingImports]
import numpy as np
from scipy.io import loadmat
from matplotlib import pyplot as plt
from google.colab import drive         # pyright: ignore[reportMissingImports]

Mount your Google Drive folders, located at `/content/drive`, on the local runtime filesystem and define the root folder for the project

In [ ]:
drive.mount('/content/drive', force_remount=True)
ROOT_FOLDER = '/content/drive/MyDrive/MMIP/data'

Set the random number generator seed for ensuring reproducibility of results 
across multiple executions

In [ ]:
np.random.seed(42)

Useful function for plot a 2D dictionary

In [ ]:
def get_dictionary_img(D):
    M, N = D.shape
    p = int(round(np.sqrt(M)))
    nnn = int(np.ceil(np.sqrt(N)))
    bound = 2
    img = np.ones((nnn*p+bound*(nnn-1), nnn*p+bound*(nnn-1)))
    for i in range(N):
        m = np.mod(i, nnn)
        n = int((i-m)/nnn)
        m = m * p + bound * m
        n = n * p + bound * n
        atom = D[:, i].reshape((p, p))
        if atom.min() < atom.max():
            atom = (atom - atom.min()) / (atom.max() - atom.min())
        img[m: m + p, n: n + p] = atom

    return img

Define a function that implements the OMP  
For convenience, we've added a utility function for input validation of OMP 
algorithm

In [ ]:
def _validate_omp_inputs(D, s, L, min_res_norm):
    if not isinstance(D, np.ndarray) or not isinstance(s, np.ndarray):
        raise TypeError("D and s must be numpy arrays!")
    if D.ndim != 2 or s.ndim != 1:
        raise ValueError(
            f"Expected D to be 2D and s to be 1D, got {D.ndim}D and {s.ndim}D.")
    if D.shape[0] != s.shape[0]:
        raise ValueError(
            f"Dimension mismatch: D rows ({D.shape[0]}) != s length ({s.shape[0]}).")
    if not isinstance(L, int) or L <= 0:
        raise ValueError("The sparsity level L must be a positive integer!")
    if min_res_norm < 0:
        raise ValueError("min_res_norm must be positive!")

In [ ]:
def OMP(s, D, L, tau):
    """
    Orthogonal Matching Pursuit (OMP) algorithm.

    Finds a sparse representation of a signal `s` with respect to a redundant 
    dictionary `D`.

    Parameters
    ----------
    D : numpy.ndarray
        A 2D array of shape (M, N) representing the redundant dictionary matrix.
    s : numpy.ndarray
        A 1D array of shape (M,) representing the signal vector.
    L : int
        The maximum allowed sparsity (number of non-zero coefficients).
    tau : float, optional
        The stopping threshold for the residual norm (default is 0.1).

    Returns
    -------
    x : numpy.ndarray
        A 1D array of shape (N,) containing the computed sparse code.

    Raises
    ------
    RuntimeError
        If a linear algebra error occurs during the update step.
    """
    _validate_omp_inputs(D, s, L, tau)

    _, N = D.shape
    x = np.zeros(N)
    r = s.copy()                    # Residual vector
    omega = []                      # Support set
    res_norm = np.linalg.norm(r)

    while np.count_nonzero(x) < L and res_norm > tau:
        # Sweep step
        e = np.zeros(N)
        for j in range(N):
            e[j] = res_norm ** 2 - (np.dot(r, D[:, j])) ** 2

        j_star = int(np.argmin(e))

        omega.append(j_star)                 # update the support set

        # Update step
        D_w = D[:, omega]

        try:
            x_w = np.linalg.solve(D_w.T @ D_w, D_w.T @ s)
        except np.linalg.LinAlgError as ex:
            raise RuntimeError(
                "Linear algebra error during update step. The matrix D_w.T @ D_w might be singular.") from ex

        x = np.zeros(N)
        x[omega] = x_w                      # update the solution
        r = s - D_w @ x_w                   # update the residual
        res_norm = np.linalg.norm(r)        # compute residual norm

    return x

Load the image and rescale it in $[0,1]$

In [ ]:
img = imread(f'{ROOT_FOLDER}/Lena512.png') / 255

imsz = img.shape
p = 8           # patch size
M = p ** 2      # no. of elements in the patch

Corrupt the image with white gaussian noise

In [ ]:
sigma_noise = 20/255
noisy_img = img + np.random.normal(0, sigma_noise, size=imsz)

Compute the PSNR of the noisy input

In [ ]:
psnr_noisy = np.log10(1/np.mean(np.square(noisy_img - img)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(img, cmap='gray')
ax[0].set_title('Original image')

ax[1].imshow(noisy_img, cmap='gray')
ax[1].set_title(f'Noisy image, PSNR = {psnr_noisy:.2f}')

Load and display the dictionary learned from patches

In [ ]:
D = loadmat(f'{ROOT_FOLDER}/dict_nat_img.mat')['D']

# Display the learned basis
D_img = get_dictionary_img(D)
plt.figure(figsize=(10, 10))
plt.imshow(D_img, cmap='gray')

Denoising
---------


In [ ]:
# Initialize the estimated image
img_hat = np.zeros_like(img)

# Initialize the weight matrix
weights = np.zeros(2*M + 1, 2*M + 1)

# Set the threshold
tau = 1.15 * p * sigma_noise

# Define the step (STEP = p for non overlapping paches)
STEP = 4  # STEP = 1 might be very time consuming, start with larger STEP

Operate patchwise

In [ ]:
for i in range(0, imsz[0] - p + 1, STEP):
    for j in range(0, imsz[1] - p + 1, STEP):
        # extrach the patch with the top left corner at pixel (ii, jj)
#        s =

        # store and subtract the mean


        # perform the sparse coding
#        x =

        # perform the reconstruction
#        s_hat =


        # add back the mean
#        s_hat =

        # put the denoised patch into the estimated image using uniform weights
#        UPDATE img_hat

        # store the weight of the current patch in the weight matrix
#        UPDATE weights

Normalize the estimated image with the computed weights

In [ ]:
# img_hat =

Compute the PSNR of the estimated image

In [ ]:
# psnr_hat =
plt.figure(figsize=(10, 10))
plt.imshow(img_hat, cmap='gray')
plt.title(f'Estimated Image,\nPSNR = {psnr_hat:.2f}')

Tikhonov Regularization Denoising
---------
Apply Tikhonov regularization and compare to OMP Denosing

In [ ]:
# initialize the estimated image for Tikhonov
img_hat_tic = np.zeros_like(img)

# initialize the weight matrix
weights_tic = np.zeros_like(img)

# set the regularization parameter
lmbda = 1

# precompute the Tikhonov matrix: x_tic = (D.T * D + lambda * I)^-1 * D.T * s
M, N = D.shape
# H_tic =

In [ ]:
for i in range(0, imsz[0] - p + 1, STEP):
    for j in range(0, imsz[1] - p + 1, STEP):
        # extract the patch with the top left corner at pixel (i, j)
#       s =


        # store and subtract the mean

        # solve using Tikhonov closed form
#       x_tic =

        # perform the reconstruction
#       s_hat =

#       w = 1

        # add back the mean
#       s_hat =

        # put the denoised patch into the estimated image using uniform weights
#       UPDATE img_hat_tic

        # store the weight of the current patch in the weight matrix
#       UPDATE weights_tic

Normalize the estimated image and evaluate the PSNR

In [ ]:
# normalize the estimated image with the computed weights
# img_hat_tic =

# compute the PSNR of the Tikhonov estimated image
# psnr_hat_tic =

fig, ax = plt.subplots(1, 2, figsize=(20, 10))

ax[0].imshow(img_hat, cmap='gray')
ax[0].set_title(f'Estimated Image (OMP Denoising),\nPSNR = {psnr_hat:.2f}')

ax[1].imshow(img_hat_tic, cmap='gray')
ax[1].set_title(f'Estimated Image (Tikhonov),\nPSNR = {psnr_hat_tic:.2f}')